# DADA3: 03, The experiment

One TODO, 3 points: the write-up at the end. The code in this notebook is given. Run each cell and read what it prints.

Your team splits the work three ways: triage, a fixer that cannot run code, and a verifier that cannot see the fix. That split costs model calls. This notebook asks whether it buys anything, by comparing the team with a **baseline**: one deep agent, with the same model and the same kind of sandbox, given all ten issues and asked to do the whole job.

Both are scored on the same things, with tests neither of them has seen.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import json, os, time
from bugteam import baseline, config, evaluate, repo
from bugteam.util import CallCounter

issues = repo.load_issues()
with open(os.path.join(config.RESULTS, "team.json")) as f:
    team_run = json.load(f)
print("team run:", team_run["model"], "|", team_run["calls"], "calls")
print("now:     ", config.model_name(), "| sandbox:", config.SANDBOX)

## 1. The baseline

Read `bugteam/baseline.py`. The baseline is one deep agent with the repository in its sandbox. Its prompt asks it to label every issue, fix the bugs, check each fix by running code, and write a report. It can do everything the team can, and more: it can run code while it writes the fix. What it lacks is a second agent checking its work.

The baseline writes its report as Markdown, and a structured-output call afterward reads its labels and its claims from that report, the way `file_ticket` read a conversation in W03b. Its fixes are read from the files it changed in the sandbox.

Run it with the same model as your team run. A baseline run is much cheaper than a team run, roughly 10 to 20 model calls, because one agent with one context does everything.

In [ ]:
assert config.model_name() == team_run["model"], "run the baseline with the model the team ran with"

counter, start = CallCounter(), time.time()
b = baseline.run(issues, callbacks=[counter])
base_run = {"model": config.model_name(), "sandbox": config.SANDBOX, "calls": counter.calls,
            "tokens": counter.tokens, "seconds": round(time.time() - start), **b}
with open(os.path.join(config.RESULTS, "baseline.json"), "w") as f:
    json.dump(base_run, f, indent=1)
print(f"{base_run['calls']} model calls, {base_run['tokens']:,} tokens, {base_run['seconds']} s, finished: {b['finished']}")
print("labels:", b["triage"])
print("claims:", b["claims"])
print("changed:", [p["path"] for p in b["patches"]])

## 2. Scoring

`bugteam/evaluate.py` is given. It applies a system's fixes to a copy of the repository in a sandbox and runs two sets of tests there:

- the repository's own tests, which should still pass
- the **hidden tests** in `eval/hidden_tests/`, which neither system has seen. For each bug, they check the case in the issue and every example in the docstring. For each issue that is not a bug, they check that the documented behavior did not change.

For the team, only the fixes the verifier accepted are applied, because those are the ones the report marks as ready. For the baseline, every file it changed is applied.

| measure | what it counts |
|---|---|
| issues resolved | out of 10: a bug whose hidden tests all pass, or a non-bug labeled correctly and left unchanged |
| triage | issues labeled correctly, out of 10 |
| bugs fixed | bugs whose hidden tests all pass, out of 7 |
| claimed | bugs the system says it fixed and checked |
| false claims | claimed fixes that the hidden tests reject |
| non-bugs unchanged | non-bugs whose documented behavior still holds, out of 3 |

A false claim is the costliest outcome of all: the report says the bug is fixed, a maintainer merges it, and the bug ships.

In [ ]:
team_score = evaluate.score_team(team_run)
base_score = evaluate.score(base_run["triage"], base_run["claims"], base_run["patches"])


def show(runs):
    for name, run, score in runs:
        t = score["totals"]
        print(f"{name:13} resolved {t['resolved']}/{t['issues']}, triage {t['triage_correct']}/{t['issues']}, "
              f"bugs fixed {t['bugs_fixed']}/{t['bugs']}, false claims {t['false_claims']}, "
              f"{run['calls']} calls, {run['tokens']:,} tokens, {run['seconds']} s")


runs = [("team", team_run, team_score), ("baseline", base_run, base_score)]
show(runs)

A second run of each system, if your quota allows. Skip this cell if it does not, and say in your write-up that you have one run of each.

In [ ]:
from bugteam import team

counter, start = CallCounter(), time.time()
out = team.run(issues, callbacks=[counter])
team_run_2 = {"model": config.model_name(), "calls": counter.calls, "tokens": counter.tokens,
              "seconds": round(time.time() - start), "labels": out["labels"], "reasons": out["reasons"], "results": out["results"]}

counter, start = CallCounter(), time.time()
b2 = baseline.run(issues, callbacks=[counter])
base_run_2 = {"model": config.model_name(), "calls": counter.calls, "tokens": counter.tokens,
              "seconds": round(time.time() - start), **b2}

with open(os.path.join(config.RESULTS, "second_runs.json"), "w") as f:
    json.dump({"team": team_run_2, "baseline": base_run_2}, f, indent=1)

runs += [("team (2)", team_run_2, evaluate.score_team(team_run_2)),
         ("baseline (2)", base_run_2, evaluate.score(base_run_2["triage"], base_run_2["claims"], base_run_2["patches"]))]
show(runs)

## 3. The handoffs

The handoff from the verifier back to the fixer is the part of the team that has no counterpart in the baseline. The cell below lists every time a verifier sent a fix back, in every team run, with the outcome of that issue in the hidden tests.

In [ ]:
for name, run, score in runs:
    if not name.startswith("team"):
        continue
    hidden = {r["issue"]: r["hidden_pass"] for r in score["rows"]}
    for r in sorted(run["results"], key=lambda r: r["number"]):
        backs = [line for line in r["log"] if line.startswith("verifier -> fixer")]
        if backs:
            print(f"{name}, #{r['number']}: {len(backs)} handback(s), final status {r['status']}, "
                  f"hidden tests {'pass' if hidden.get(r['number']) else 'fail'}")
            for line in backs:
                print("   ", line[:400].replace("\n", " "))
            print()

Read each handback above, and check the verifier's reason against the docstring. Did the fix it sent back really fail something the docstring promises, or did the verifier test something the docstring does not say? Did the next fix address the reason? The write-up asks about one of them.

## 4. Where each system went wrong

The cell below lists every issue on which a run was wrong: a wrong triage label, a bug not fixed, a false claim, or a non-bug whose behavior changed. For the team it prints the verifier's test, and for the baseline the part of its report about that issue.

In [ ]:
labels = evaluate.load_labels()
for name, run, score in runs:
    results = {r["number"]: r for r in run.get("results", [])}
    for row_ in score["rows"]:
        wrong = (not row_["triage_ok"]) or row_["false_claim"] or not row_["hidden_pass"]
        if not wrong:
            continue
        print(f"== {name}, #{row_['issue']} ({row_['label']}): triage said {row_['triage']}, "
              f"claimed fixed {row_['claimed_fixed']}, hidden tests {'pass' if row_['hidden_pass'] else 'fail'}")
        if name.startswith("team") and row_["issue"] in results:
            r = results[row_["issue"]]
            print("verifier:", r["evidence"][:500])
            print(r["test"][:1500] or "(no test)")
        elif name.startswith("baseline"):
            text = run.get("report", "")
            at = text.find(f"#{row_['issue']}")
            print(text[at:at + 800] if at >= 0 else "(the report does not mention this issue)")
        print()

For each failure above, read the test or the report behind it. When the team made a false claim, the verifier's test missed something: an example from the docstring it left out, an example it changed so that the fix would pass, or a case it never thought of.

### TODO 3.1: the write-up (3 pts)

Write this in `README.md`, under the headings given there.

1. **Setup.** The model, the sandbox, how many runs of each system, and anything you changed from the bundle as given, with the reason.
2. **Results.** The numbers printed in section 2, for every run.
3. **Did the team do better?** Compare issues resolved and false claims. With ten issues, one issue is 10 points of the percentage resolved, so say whether the difference is larger than the difference between two runs of the same system.
4. **The handoff.** One handback from your runs: the verifier's reason, whether it was right by the docstring, and whether the next fix passed. If there were none, say why, from the logs.
5. **What did it cost?** Model calls and tokens for each system, and whether the difference is worth what the team bought.

You are not graded on the team winning. A write-up that shows, with your numbers and transcripts, that the baseline did as well for a fifth of the cost is worth as much as one that shows the team doing better.